# SIFT (Scale-Invariant Feature Transform) — Tổng quan thuật toán

**SIFT** là thuật toán trích xuất đặc trưng ảnh bất biến với phép tỷ lệ (scale), phép quay (rotation) và tương đối bền với thay đổi ánh sáng, được David Lowe công bố năm 1999/2004. SIFT gồm **4 bước chính**:

### 1. Scale-space Extrema Detection (Phát hiện cực trị trong không gian tỷ lệ)
- Xây dựng **Gaussian Pyramid**: làm mờ ảnh gốc bằng Gaussian Blur với độ lệch chuẩn (σ) tăng dần qua nhiều "scale", đồng thời giảm kích thước ảnh (downsample) qua nhiều "octave".
- Từ Gaussian Pyramid, tính **Difference of Gaussian (DoG) Pyramid** bằng cách lấy hiệu của các ảnh Gaussian liền kề nhau — đây là phép xấp xỉ của Laplacian of Gaussian (LoG), dùng để phát hiện các điểm đặc trưng ở nhiều tỷ lệ khác nhau.
- Tìm **cực trị cục bộ** (local maxima/minima) trong khối 3×3×3 (so sánh 1 điểm với 26 điểm lân cận: 8 điểm cùng scale + 9 điểm scale trên + 9 điểm scale dưới).

### 2. Keypoint Localization (Định vị và lọc điểm đặc trưng)
- Tinh chỉnh vị trí keypoint bằng nội suy bậc 2 (Taylor expansion) để đạt độ chính xác dưới-pixel (sub-pixel).
- Loại bỏ các điểm có độ tương phản thấp (low contrast → dễ bị nhiễu).
- Loại bỏ các điểm nằm trên cạnh (edge response) bằng cách kiểm tra tỉ số giá trị riêng của ma trận Hessian.

### 3. Orientation Assignment (Gán hướng chủ đạo)
- Với mỗi keypoint, tính gradient (độ lớn và hướng) trong vùng lân cận.
- Xây dựng histogram hướng gradient (36 bin, mỗi bin 10°), hướng có tần suất cao nhất (và các hướng ≥ 80% giá trị đỉnh) được gán làm hướng chủ đạo → giúp keypoint bất biến với phép quay.

### 4. Keypoint Descriptor (Xây dựng vector đặc trưng)
- Quanh mỗi keypoint, lấy vùng lân cận 16×16 pixel, chia thành 16 ô 4×4.
- Mỗi ô tính histogram gradient 8 hướng → tổng cộng vector đặc trưng **4×4×8 = 128 chiều**, sau đó chuẩn hoá để tăng độ bền với thay đổi ánh sáng.

---

## Phạm vi của file `V1_gpu_naive.py`

File này **chỉ tập trung tăng tốc Bước 1 — phần xây dựng Gaussian Pyramid** (phần tốn kém nhất về mặt tính toán vì phải convolve ảnh nhiều lần với kernel Gaussian ở nhiều scale/octave), bằng cách:

- Cài đặt Gaussian Blur dưới dạng **Separable 1D Convolution** (blur theo hàng rồi blur theo cột, thay vì convolution 2D trực tiếp) để giảm độ phức tạp tính toán từ O(k²) xuống O(2k) với k là kích thước kernel.
- Chạy các phép convolution này trên **GPU bằng Numba CUDA**, đây là phiên bản **"naive" (ngây thơ)** — mỗi thread GPU xử lý đúng 1 pixel đầu ra và đọc dữ liệu trực tiếp từ **Global Memory** (chưa dùng Shared Memory hay các kỹ thuật tối ưu bộ nhớ khác). Đây thường là bước "V1" trong chuỗi tối ưu dần (V0 CPU → V1 GPU naive → V2 GPU shared memory → ...).
- Kết quả GPU được so sánh (verify) với bản CPU dùng `scipy` (từ file `V0_1_with_lib.py`) để đảm bảo tính đúng đắn, và benchmark tốc độ giữa CPU và GPU.


## Phần 1: Import thư viện và thiết lập môi trường

Đoạn code dưới đây import các thư viện cần thiết:
- `sys`, `os`, `time`, `math`: thư viện chuẩn của Python để thao tác hệ thống, đo thời gian, tính toán.
- `numpy as np`: xử lý mảng số học (ảnh được biểu diễn dưới dạng mảng NumPy).
- `glob`: tìm kiếm file theo pattern (dùng để lấy danh sách ảnh DIV2K).
- `numba.cuda`: module chính cho phép viết CUDA kernel bằng cú pháp Python (JIT compile sang mã máy chạy trên GPU).
- `cv2` (OpenCV): chỉ dùng để đọc/ghi ảnh và chạy SIFT tham chiếu (`cv2.SIFT_create()`), được bọc trong `try/except` để chương trình không bị crash nếu môi trường thiếu OpenCV.

Dòng `sys.stdout.reconfigure(encoding="utf-8")` ép terminal (đặc biệt là Windows) hiển thị đúng tiếng Việt có dấu.


In [1]:
import sys
import os
import time
import math
import numpy as np
import glob

# Ep UTF-8 cho Windows terminal
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

from numba import cuda

# OpenCV chi dung de load anh
try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False


### Mount Google Drive và nạp module V0 (bản CPU tham chiếu)

Vì code này được chạy trên **Google Colab**, nó cần:
1. `drive.mount(...)`: gắn (mount) Google Drive của người dùng vào `/content/drive` để có thể đọc file ảnh DIV2K và file mã nguồn `V0_1_with_lib.py` được lưu trên Drive.
2. Thêm thư mục chứa `V0_1_with_lib.py` vào đầu `sys.path` (loại bỏ path cũ nếu đã tồn tại rồi chèn lên đầu) để đảm bảo Python import đúng phiên bản module mới nhất, tránh dùng bản cache cũ.
3. `importlib.invalidate_caches()` + `importlib.util.find_spec(...)`: xoá cache import của Python và kiểm tra xem module `V0_1_with_lib` có thực sự được tìm thấy tại đường dẫn mong muốn hay không (in ra `spec` và `origin` để debug).
4. `import V0_1_with_lib`: import module CPU tham chiếu, dùng để đối chiếu kết quả (verify) và benchmark so sánh CPU vs GPU.

Sau đó, các hàm cụ thể được import trực tiếp từ `V0_1_with_lib`:
- `build_gaussian_pyramid_with_lib`: hàm CPU xây Gaussian Pyramid (dùng `scipy`), làm chuẩn đối chiếu.
- `compute_dog_pyramid`: tính DoG Pyramid từ Gaussian Pyramid (bước 1 của SIFT).
- `create_synthetic_div2k_image`, `load_single_image`, `get_div2k_image_paths`: các hàm tiện ích để tạo ảnh giả lập hoặc nạp ảnh thật từ bộ dữ liệu DIV2K.

Nếu import thất bại (`ImportError`), cờ `V0_AVAILABLE = False` được set để các phần benchmark/verify có thể được bỏ qua an toàn.


In [2]:
import importlib.util
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

folder_path = '/content/drive/MyDrive/Colab Notebooks'
print("folder exists:", os.path.isdir(folder_path))

# loại bản path cũ (nếu có) rồi chèn lên đầu
sys.path = [p for p in sys.path if p != folder_path]
sys.path.insert(0, folder_path)

print("sys.path[0]:", sys.path[0])
print("file exists:", os.path.isfile(os.path.join(folder_path, 'V0_1_with_lib.py')))

importlib.invalidate_caches()
spec = importlib.util.find_spec("V0_1_with_lib")
print("spec:", spec)
print("origin:", spec.origin if spec else None)

import V0_1_with_lib
print("IMPORT OK:", V0_1_with_lib.__file__)

# Import tu V0 de verify va benchmark
try:
    from V0_1_with_lib import (
        build_gaussian_pyramid_with_lib,
        compute_dog_pyramid,
        create_synthetic_div2k_image,
        load_single_image,
        get_div2k_image_paths
    )
    V0_AVAILABLE = True
except ImportError:
    V0_AVAILABLE = False
    print("[ERROR] Khong tim thay V0_1_with_lib.py - bo qua tinh nang benchmark.")


Mounted at /content/drive
folder exists: True
sys.path[0]: /content/drive/MyDrive/Colab Notebooks
file exists: True
spec: ModuleSpec(name='V0_1_with_lib', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7d95f05fc950>, origin='/content/drive/MyDrive/Colab Notebooks/V0_1_with_lib.py')
origin: /content/drive/MyDrive/Colab Notebooks/V0_1_with_lib.py
IMPORT OK: /content/drive/MyDrive/Colab Notebooks/V0_1_with_lib.py


## Phần 2: CUDA Kernels cho Gaussian Blur (phiên bản Naive)

Đây là phần lõi kỹ thuật của file. Ý tưởng: thay vì làm convolution 2D trực tiếp (tốn O(k²) phép nhân cho mỗi pixel với k là kích thước kernel), ta tách Gaussian 2D thành **2 lượt convolution 1D độc lập** (theo hàng rồi theo cột) — gọi là **Separable Convolution** — giảm độ phức tạp xuống O(2k) mỗi pixel. Toán học đúng vì Gaussian 2D = tích của 2 Gaussian 1D (theo x và theo y).


### Hàm `reflect_idx(i, n)` — Device function

**Là gì:** một *device function* (`@cuda.jit(device=True, inline=True)`) — tức là hàm chỉ chạy **trên GPU**, được gọi từ bên trong các kernel khác (không thể gọi trực tiếp từ CPU/host). `inline=True` yêu cầu trình biên dịch chèn thẳng code của hàm vào nơi gọi để giảm overhead gọi hàm.

**Làm gì:** xử lý điều kiện biên (boundary condition) khi chỉ số pixel `i` vượt ra ngoài khoảng hợp lệ `[0, n)` (n là chiều rộng hoặc chiều cao ảnh), theo kiểu **"reflect"** giống với chế độ `mode='reflect'` của `scipy.ndimage` — tức ảnh được "phản chiếu" qua biên thay vì lặp lại (`wrap`) hoặc lấy giá trị biên (`nearest`).

**Cách hoạt động (từng dòng):**
- `while i < 0 or i >= n:` — lặp cho đến khi `i` nằm trong khoảng hợp lệ (dùng vòng lặp vì với kernel lớn, chỉ số có thể "văng" ra ngoài nhiều lần liên tiếp, cần phản chiếu qua lại nhiều lượt mới về đúng khoảng).
- `if i < 0: i = -i - 1` — nếu `i` âm, phản chiếu qua biên trái: ví dụ `i = -1` → `0`, `i = -2` → `1`,... (phản chiếu không lặp lại điểm biên, giống scipy).
- `elif i >= n: i = 2 * n - i - 1` — nếu `i` vượt quá biên phải, phản chiếu qua biên phải theo công thức đối xứng.
- `return i` — trả về chỉ số hợp lệ sau khi phản chiếu xong.


In [3]:
@cuda.jit(device=True, inline=True)
def reflect_idx(i, n):
    # phản chiếu kiểu scipy reflect (gần đúng thực tế dùng trong gaussian_filter)
    # an toàn cho nhiều vòng vượt biên
    while i < 0 or i >= n:
        if i < 0:
            i = -i - 1
        elif i >= n:
            i = 2 * n - i - 1
    return i


### Kernel `gaussian_blur_row_kernel(d_input, d_output, d_kernel, radius)`

**Là gì:** một CUDA kernel thật sự (`@cuda.jit`, không có `device=True`) — được **gọi từ host (CPU)** nhưng thực thi song song trên hàng nghìn thread GPU. Đây là lượt convolution 1D **theo chiều ngang (hàng / row)**.

**Làm gì:** với mỗi pixel đầu ra `(x, y)`, tính tổng có trọng số (weighted sum) của các pixel lân cận theo trục ngang trong bán kính `radius`, dùng kernel Gaussian 1D `d_kernel`.

**Cách hoạt động (từng dòng):**
- `x, y = cuda.grid(2)` — lấy toạ độ toàn cục (global index) của thread hiện tại trong lưới 2D. Mỗi thread GPU tương ứng đúng 1 pixel đầu ra `(x, y)` — đây chính là đặc điểm "naive": **1 thread ↔ 1 pixel output**, không có sự chia sẻ công việc/tái sử dụng dữ liệu giữa các thread.
- `H, W = d_input.shape` — lấy chiều cao/rộng ảnh đầu vào.
- `if x >= W or y >= H: return` — kiểm tra biên: vì số thread được cấp phát theo bội số của block size nên có thể dư ra ngoài kích thước ảnh thật, các thread thừa này thoát sớm không làm gì.
- `acc = 0.0` — biến tích luỹ (accumulator) kết quả convolution, khởi tạo bằng 0.
- `for i in range(-radius, radius + 1):` — duyệt qua toàn bộ kernel Gaussian 1D, từ `-radius` đến `+radius`.
- `idx = reflect_idx(x + i, W)` — tính chỉ số cột thực tế cần đọc (`x + i`), xử lý biên bằng hàm `reflect_idx` đã định nghĩa ở trên.
- `acc += d_input[y, idx] * d_kernel[i + radius]` — đọc **trực tiếp từ Global Memory** (`d_input`) giá trị pixel tại `(y, idx)` rồi nhân với trọng số Gaussian tương ứng, cộng dồn vào `acc`. Đây chính là điểm "ngây thơ" (naive) của thiết kế: mỗi thread tự đọc lại dữ liệu từ Global Memory (chậm hơn nhiều so với Shared Memory), và các thread lân cận đọc trùng lặp nhiều pixel giống nhau mà không chia sẻ.
- `d_output[y, x] = acc` — ghi kết quả convolution theo hàng vào ảnh đầu ra.


In [ ]:
@cuda.jit
def gaussian_blur_row_kernel(d_input, d_output, d_kernel, radius):
    x, y = cuda.grid(2)
    H, W = d_input.shape
    if x >= W or y >= H:
        return

    acc = 0.0
    for i in range(-radius, radius + 1):
        idx = reflect_idx(x + i, W)
        acc += d_input[y, idx] * d_kernel[i + radius]
    d_output[y, x] = acc


### Kernel `gaussian_blur_col_kernel(d_input, d_output, d_kernel, radius)`

**Là gì:** CUDA kernel giống hệt cấu trúc với `gaussian_blur_row_kernel` ở trên, nhưng là lượt convolution 1D **theo chiều dọc (cột / column)** — lượt thứ 2 của separable convolution.

**Làm gì:** nhận đầu vào là kết quả của lượt blur theo hàng (`d_temp` ở hàm gọi bên dưới), tiếp tục làm mờ theo chiều dọc để hoàn tất phép Gaussian Blur 2D đầy đủ.

**Cách hoạt động (từng dòng):** hoàn toàn tương tự kernel trên, chỉ khác ở chiều duyệt:
- `x, y = cuda.grid(2)` và kiểm tra biên `if x >= W or y >= H: return` giống hệt kernel theo hàng.
- `for i in range(-radius, radius + 1): idy = reflect_idx(y + i, H)` — lần này chỉ số bị dịch chuyển và phản chiếu là **chỉ số hàng** `y + i` (thay vì cột), vì đang convolve theo trục dọc.
- `acc += d_input[idy, x] * d_kernel[i + radius]` — đọc pixel tại `(idy, x)` (cùng cột `x`, khác hàng), nhân với trọng số kernel rồi cộng dồn.
- `d_output[y, x] = acc` — ghi kết quả cuối cùng (đã blur cả 2 chiều) vào ảnh đầu ra.


In [ ]:
@cuda.jit
def gaussian_blur_col_kernel(d_input, d_output, d_kernel, radius):
    x, y = cuda.grid(2)
    H, W = d_input.shape
    if x >= W or y >= H:
        return

    acc = 0.0
    for i in range(-radius, radius + 1):
        idy = reflect_idx(y + i, H)
        acc += d_input[idy, x] * d_kernel[i + radius]
    d_output[y, x] = acc


### Hàm `make_gaussian_kernel_1d(sigma, truncate=4.0)`

**Là gì:** hàm Python thuần (chạy trên **CPU**, không phải CUDA kernel) dùng để tạo trước (precompute) mảng trọng số Gaussian 1D, sau đó mảng này được copy sang GPU (`d_kernel`) để 2 kernel ở trên sử dụng.

**Làm gì:** sinh ra kernel Gaussian 1D chuẩn hoá (tổng các phần tử = 1), với cách tính bán kính giống hệt cách `scipy.ndimage.gaussian_filter` sử dụng (để đảm bảo kết quả GPU khớp với kết quả CPU tham chiếu khi verify).

**Cách hoạt động (từng dòng):**
- `radius = int(truncate * float(sigma) + 0.5)` — tính bán kính kernel dựa trên `sigma` (độ lệch chuẩn) và hệ số cắt `truncate` (mặc định 4.0, nghĩa là kernel trải rộng ±4σ) — đây chính xác là công thức mà `scipy` dùng nội bộ, nên kích thước kernel giữa 2 phiên bản sẽ khớp nhau.
- `x = np.arange(-radius, radius + 1, dtype=np.float32)` — tạo mảng chỉ số từ `-radius` đến `radius` (tổng cộng `2*radius + 1` phần tử).
- `k1d = np.exp(-x * x / (2.0 * sigma * sigma))` — áp dụng công thức hàm mật độ Gaussian.
- `k1d /= k1d.sum()` — chuẩn hoá để tổng các trọng số bằng 1, đảm bảo phép convolution không làm thay đổi độ sáng trung bình của ảnh.
- `return k1d.astype(np.float32)` — ép kiểu về `float32` để khớp với kiểu dữ liệu ảnh trên GPU (tiết kiệm bộ nhớ và tăng tốc so với `float64`).



In [6]:
def make_gaussian_kernel_1d(sigma, truncate=4.0):
    """
    Tao 1D Gaussian kernel giong voi cach scipy thuc hien (truncate=4.0 default).
    """
    radius = int(truncate * float(sigma) + 0.5)
    x = np.arange(-radius, radius + 1, dtype=np.float32)
    k1d = np.exp(-x * x / (2.0 * sigma * sigma))
    k1d /= k1d.sum()
    return k1d.astype(np.float32)


### Hàm `build_gaussian_pyramid_gpu_v1(image, num_octaves=4, num_scales=5, sigma_base=1.6)`

**Là gì:** hàm điều phối chính chạy trên **host (CPU)**, có nhiệm vụ gọi 2 CUDA kernel ở trên theo đúng trình tự để xây dựng toàn bộ **Gaussian Pyramid** — cấu trúc dữ liệu cốt lõi ở Bước 1 của SIFT — cho một ảnh đầu vào.

**Làm gì:** với mỗi octave (mức tỷ lệ ảnh, ảnh sau mỗi octave có kích thước bằng 1/2 octave trước), và với mỗi scale trong octave đó (mức độ mờ tăng dần), hàm thực hiện: (1) tạo kernel Gaussian tương ứng, (2) chạy 2 kernel GPU để blur theo hàng rồi theo cột (separable convolution), (3) lưu kết quả, sau đó downsample ảnh để chuẩn bị cho octave kế tiếp.

**Cách hoạt động (từng dòng/khối):**
- `pyramid = []` — danh sách chứa toàn bộ pyramid, mỗi phần tử là 1 octave (list các ảnh đã blur ở các scale khác nhau).
- `current_img = image.astype(np.float32)` và `d_current_img = cuda.to_device(current_img)` — chuyển ảnh đầu vào sang `float32` và copy từ RAM (host) sang bộ nhớ GPU (device) — đây là bước copy Host→Device bắt buộc trước khi GPU có thể xử lý.
- `k = 2.0 ** (1.0 / num_scales)` — hệ số nhân sigma giữa các scale liên tiếp trong cùng 1 octave, theo đúng công thức chuẩn của SIFT (chia đều `num_scales` bước trong khoảng tăng gấp đôi của sigma).
- `block_dim = (16, 16)` — kích thước block CUDA cố định 16×16 = 256 thread/block, một lựa chọn phổ biến cân bằng giữa occupancy và tài nguyên GPU.
- **Vòng lặp `for octave in range(num_octaves):`**
  - `H, W = d_current_img.shape` — lấy kích thước ảnh hiện tại của octave.
  - `grid_dim = (ceil(W/16), ceil(H/16))` — tính số block cần thiết theo cả 2 chiều để phủ hết toàn bộ ảnh (làm tròn lên bằng `math.ceil` để không thiếu pixel biên).
  - **Vòng lặp `for scale in range(num_scales):`**
    - `sigma = sigma_base * (k ** scale)` — sigma tăng dần theo cấp số nhân qua từng scale, bắt đầu từ `sigma_base` (mặc định 1.6, giá trị chuẩn trong bài báo gốc của SIFT).
    - `k1d = make_gaussian_kernel_1d(sigma)` — sinh kernel Gaussian 1D tương ứng với sigma này (trên CPU).
    - `radius = len(k1d) // 2` — tính bán kính từ độ dài kernel.
    - `d_kernel = cuda.to_device(k1d)` — copy kernel sang GPU.
    - `d_temp = cuda.device_array((H, W), dtype=np.float32)` và `d_blurred = ...` — cấp phát 2 mảng trống trên GPU: `d_temp` chứa kết quả sau lượt blur theo hàng, `d_blurred` chứa kết quả cuối sau lượt blur theo cột.
    - `gaussian_blur_row_kernel[grid_dim, block_dim](...)` — gọi kernel blur theo hàng, cú pháp `[grid_dim, block_dim]` chỉ định cấu hình launch (bao nhiêu block, bao nhiêu thread/block).
    - `gaussian_blur_col_kernel[grid_dim, block_dim](...)` — gọi kernel blur theo cột, nhận đầu vào là `d_temp` (kết quả bước trước) → hoàn tất separable convolution 2D.
    - `octave_imgs.append(d_blurred)` — lưu con trỏ ảnh (vẫn còn trên GPU) vào danh sách của octave hiện tại.
  - `host_octave = [d_img.copy_to_host() for d_img in octave_imgs]` — sau khi xử lý xong toàn bộ scale trong octave, copy tất cả ảnh kết quả từ Device về Host (RAM) một lần.
  - `pyramid.append(host_octave)` — thêm octave vừa hoàn thành vào pyramid tổng.
  - `mid_img = host_octave[num_scales // 2]` — lấy ảnh ở scale giữa (middle scale) của octave hiện tại làm cơ sở cho octave tiếp theo (theo đúng quy ước chuẩn của SIFT).
  - `current_img = np.ascontiguousarray(mid_img[::2, ::2])` — **downsample** ảnh bằng cách lấy mẫu cách 1 pixel theo cả 2 chiều (`[::2, ::2]`), giảm kích thước ảnh đi một nửa mỗi chiều để chuẩn bị cho octave kế tiếp; `np.ascontiguousarray` đảm bảo mảng liên tục trong bộ nhớ (cần thiết trước khi copy sang GPU).
  - `d_current_img = cuda.to_device(current_img)` — copy ảnh đã downsample lên GPU, sẵn sàng cho vòng lặp octave kế tiếp.
- `return pyramid` — trả về toàn bộ Gaussian Pyramid dưới dạng list-of-lists các mảng NumPy (đã ở trên host).


In [ ]:
def build_gaussian_pyramid_gpu_v1(image, num_octaves=4, num_scales=5, sigma_base=1.6):
    """
    Xay dung Gaussian Pyramid tren GPU (V1 Naive).
    Giung kernel doc truc tiep tu global memory, khong dung shared memory.
    """
    pyramid = []

    # 1. Chuan bi du lieu (Host -> Device)
    current_img = image.astype(np.float32)
    d_current_img = cuda.to_device(current_img)

    k = 2.0 ** (1.0 / num_scales)
    block_dim = (16, 16)

    for octave in range(num_octaves):
        octave_imgs = []
        H, W = d_current_img.shape
        grid_dim = (int(math.ceil(W / block_dim[0])), int(math.ceil(H / block_dim[1])))

        for scale in range(num_scales):
            sigma = sigma_base * (k ** scale)
            k1d = make_gaussian_kernel_1d(sigma)
            radius = len(k1d) // 2

            d_kernel = cuda.to_device(k1d)
            d_temp = cuda.device_array((H, W), dtype=np.float32)
            d_blurred = cuda.device_array((H, W), dtype=np.float32)

            # 2. Goi CUDA kernels (Device execution)
            gaussian_blur_row_kernel[grid_dim, block_dim](d_current_img, d_temp, d_kernel, radius)
            gaussian_blur_col_kernel[grid_dim, block_dim](d_temp, d_blurred, d_kernel, radius)

            # Trong SIFT tieu chuan, moi scale dua tren anh truoc do hoac anh goc hien tai,
            # o day ta tua theo phien ban reference la blur rieng le tung anh.
            octave_imgs.append(d_blurred)

        # 3. Copy ket qua tu Device -> Host de downsample
        host_octave = [d_img.copy_to_host() for d_img in octave_imgs]
        pyramid.append(host_octave)

        # 4. Downsample tren Host va dua len Device cho octave tiep theo
        mid_img = host_octave[num_scales // 2]
        current_img = np.ascontiguousarray(mid_img[::2, ::2])
        d_current_img = cuda.to_device(current_img)

    return pyramid


## Phần 3: Kiểm tra độ chính xác (Verify) và Benchmark

Phần này dùng để (a) đảm bảo kết quả GPU đúng so với CPU tham chiếu, và (b) đo tốc độ thực thi để tính speedup.

### Hàm `verify_correctness(img, num_octaves, num_scales)`

**Là gì:** hàm kiểm thử độ chính xác, chạy trên CPU (chỉ gọi tới các hàm build pyramid, không tự làm phép tính ảnh).

**Làm gì:** xây dựng Gaussian Pyramid bằng cả 2 phương pháp — CPU tham chiếu (`build_gaussian_pyramid_with_lib`, dùng `scipy`) và GPU naive (`build_gaussian_pyramid_gpu_v1`) — trên cùng một ảnh đầu vào, rồi so sánh sai số tuyệt đối lớn nhất (Max Absolute Error, biến được đặt tên là `err`/`max_err`, dù comment ghi "MAE" nhưng thực chất tính bằng `.max()` chứ không phải trung bình) giữa từng cặp ảnh tương ứng.

**Cách hoạt động (từng dòng):**
- `ref_pyramid = build_gaussian_pyramid_with_lib(...)` — sinh pyramid chuẩn từ CPU (scipy).
- `gpu_pyramid = build_gaussian_pyramid_gpu_v1(...)` — sinh pyramid từ GPU naive.
- `max_err = 0.0`, `passed = True` — khởi tạo biến theo dõi sai số lớn nhất và trạng thái pass/fail.
- Vòng lặp lồng `for o in range(num_octaves): for s in range(num_scales):` — duyệt qua từng cặp ảnh `(octave, scale)` tương ứng giữa 2 pyramid.
  - `err = np.abs(ref - out).max()` — tính sai số tuyệt đối lớn nhất giữa ảnh CPU (`ref`) và ảnh GPU (`out`) tại vị trí `(o, s)`.
  - `max_err = max(max_err, err)` — cập nhật sai số lớn nhất toàn cục.
  - `if err > 1e-3:` — nếu sai số vượt ngưỡng cho phép (1e-3), in ra cảnh báo lỗi cụ thể tại octave/scale nào và đặt `passed = False`.
- Cuối cùng in ra kết quả tổng kết: `[PASS]` nếu mọi layer đều đạt ngưỡng sai số, hoặc `[FAIL]` kèm giá trị sai số lớn nhất nếu có ít nhất 1 layer vượt ngưỡng.


In [ ]:
def verify_correctness(img, num_octaves, num_scales):
    """Kiem tra do chinh xac (MAE) giua GPU V1 va CPU (scipy)."""
    print("\n[VERIFY] Kiem tra do chinh xac MAE (GPU vs CPU scipy)...")

    ref_pyramid = build_gaussian_pyramid_with_lib(img, num_octaves, num_scales)
    gpu_pyramid = build_gaussian_pyramid_gpu_v1(img, num_octaves, num_scales)

    max_err = 0.0
    passed = True

    for o in range(num_octaves):
        for s in range(num_scales):
            ref = ref_pyramid[o][s]
            out = gpu_pyramid[o][s]

            err = np.abs(ref - out).max()
            max_err = max(max_err, err)

            if err > 1e-3:
                print(f"  [ERROR] Octave {o}, Scale {s}: MAE = {err:.5e} > 1e-3")
                passed = False

    if passed:
        print(f"  [PASS] Tat ca layers co MAE < 1e-3 (Max MAE = {max_err:.5e})")
    else:
        print(f"  [FAIL] Do chinh xac khong dat yeu cau (Max MAE = {max_err:.5e})")


### Hàm `benchmark_single(build_fn, image, num_octaves, num_scales, n_runs=3, label="")`

**Là gì:** hàm tiện ích đo thời gian thực thi trung bình của một hàm xây dựng pyramid bất kỳ (`build_fn` — có thể là bản CPU hoặc GPU), giúp code benchmark không bị lặp lại giữa các phiên bản.

**Làm gì:** chạy `build_fn` một lần "khởi động" (warmup) trước để loại bỏ chi phí biên dịch JIT lần đầu (đặc biệt quan trọng với Numba CUDA vì lần gọi đầu tiên sẽ compile kernel sang mã máy, tốn thời gian không phản ánh hiệu năng thực), sau đó chạy `n_runs` lần và lấy trung bình.

**Cách hoạt động (từng dòng):**
- `build_fn(image, num_octaves, num_scales)` (dòng đầu, ngoài vòng lặp) — lượt **warmup**: chạy 1 lần không tính giờ, để Numba JIT-compile các kernel CUDA trước; nếu không có bước này, lần đo đầu tiên sẽ bị "ăn" chi phí compile khiến kết quả benchmark sai lệch.
- `times = []` — danh sách lưu thời gian từng lượt chạy.
- Vòng lặp `for _ in range(n_runs):`
  - `t0 = time.perf_counter()` — ghi mốc thời gian bắt đầu (dùng `perf_counter` vì có độ phân giải cao, phù hợp đo hiệu năng).
  - `build_fn(image, num_octaves, num_scales)` — thực thi hàm cần đo.
  - `cuda.synchronize() if hasattr(cuda, 'synchronize') else None` — đồng bộ hoá GPU: vì các lệnh gọi kernel CUDA là **bất đồng bộ** (asynchronous, hàm Python trả về ngay dù GPU chưa chạy xong), bắt buộc phải gọi `cuda.synchronize()` để đảm bảo GPU thực sự hoàn thành công việc trước khi đo thời gian kết thúc — nếu thiếu bước này, thời gian đo được sẽ không chính xác (đo thiếu).
  - `times.append(time.perf_counter() - t0)` — tính và lưu thời gian chạy của lượt này.
- `return float(np.mean(times))` — trả về thời gian trung bình qua `n_runs` lượt chạy.


In [ ]:
def benchmark_single(build_fn, image, num_octaves, num_scales, n_runs=3, label=""):
    """Chay nhieu lan lay trung binh."""
    # Warmup
    build_fn(image, num_octaves, num_scales)

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        build_fn(image, num_octaves, num_scales)
        # Bua sau co the bat de benchmark ca compute_dog, nhung tam thoi bo qua de giong V0
        # dog = compute_dog_pyramid(pyramid)
        cuda.synchronize() if hasattr(cuda, 'synchronize') else None
        times.append(time.perf_counter() - t0)

    return float(np.mean(times))


## Phần 4: Hoàn thiện SIFT bằng OpenCV (tham chiếu keypoint thực tế)

Vì file này chỉ tăng tốc **Bước 1** của SIFT (Gaussian Pyramid), 3 bước còn lại (DoG extrema, orientation, descriptor) được **thay thế bằng cài đặt SIFT có sẵn của OpenCV** để lấy kết quả keypoint thực tế phục vụ minh hoạ/lưu ảnh kết quả, thay vì tự cài đặt lại toàn bộ.

### Hàm `run_sift_opencv(gray, max_detect_dim=1200)`

**Là gì:** hàm chạy trên CPU, dùng module `cv2.SIFT_create()` của OpenCV để phát hiện keypoint và tính descriptor SIFT đầy đủ (toàn bộ 4 bước) trên một ảnh xám.

**Làm gì:** để tăng tốc độ phát hiện, ảnh được resize xuống nếu quá lớn (giới hạn `max_detect_dim`) trước khi chạy SIFT, sau đó toạ độ/kích thước các keypoint tìm được sẽ được **scale ngược lại** về đúng kích thước ảnh gốc.

**Cách hoạt động (từng dòng):**
- `h, w = gray.shape` — lấy kích thước ảnh xám đầu vào.
- `scale = 1.0`, `gray_detect = gray` — mặc định không resize, dùng chính ảnh gốc để detect.
- `if max_detect_dim is not None and max(h, w) > max_detect_dim:` — nếu cạnh lớn nhất của ảnh vượt quá ngưỡng cho phép:
  - `scale = max_detect_dim / max(h, w)` — tính hệ số thu nhỏ sao cho cạnh lớn nhất đúng bằng `max_detect_dim`.
  - `gray_detect = cv2.resize(...)` — resize ảnh xuống kích thước nhỏ hơn bằng nội suy `INTER_AREA` (phù hợp khi thu nhỏ ảnh, giảm hiện tượng aliasing) để việc detect SIFT nhanh hơn trên ảnh có độ phân giải cao.
- `t0 = time.time()` — bắt đầu đo thời gian.
- `sift = cv2.SIFT_create()` — khởi tạo detector SIFT của OpenCV với tham số mặc định.
- `kp, des = sift.detectAndCompute(gray_detect, None)` — chạy toàn bộ pipeline SIFT (phát hiện + tính descriptor) trên ảnh đã resize; trả về `kp` (danh sách keypoint) và `des` (ma trận descriptor 128 chiều cho mỗi keypoint).
- `elapsed = time.time() - t0` — tính thời gian chạy SIFT.
- `if scale != 1.0:` — nếu có resize trước đó, cần quy đổi toạ độ keypoint về đúng ảnh gốc:
  - `inv_scale = 1.0 / scale` — hệ số nghịch đảo để phóng to lại toạ độ.
  - Danh sách `kp` được tạo lại bằng list comprehension: mỗi `cv2.KeyPoint` mới có toạ độ `x, y` và `size` (kích thước vùng đặc trưng) được nhân với `inv_scale`, còn `angle` (hướng), `response` (độ mạnh phản hồi), `octave`, `class_id` được giữ nguyên vì đây là các thuộc tính bất biến tỷ lệ.
- `return kp, des, elapsed, scale` — trả về keypoint (đã quy đổi đúng tỷ lệ ảnh gốc), descriptor, thời gian chạy, và hệ số scale đã dùng.


In [10]:
def run_sift_opencv(gray: np.ndarray, max_detect_dim=1200):
    h, w = gray.shape
    scale = 1.0
    gray_detect = gray

    if max_detect_dim is not None and max(h, w) > max_detect_dim:
        scale = max_detect_dim / max(h, w)
        gray_detect = cv2.resize(gray, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)

    t0 = time.time()
    sift = cv2.SIFT_create()
    kp, des = sift.detectAndCompute(gray_detect, None)
    elapsed = time.time() - t0

    if scale != 1.0:
        inv_scale = 1.0 / scale
        kp = [
            cv2.KeyPoint(x=k.pt[0] * inv_scale, y=k.pt[1] * inv_scale, size=k.size * inv_scale,
                         angle=k.angle, response=k.response, octave=k.octave, class_id=k.class_id)
            for k in kp
        ]
    return kp, des, elapsed, scale


## Phần 5: Hàm `main()` — Điều phối toàn bộ chương trình



In [11]:
def main():
    print("=" * 70)
    print("  V1_gpu_naive.py -- GPU V1 (Naive Numba CUDA)")
    print("=" * 70)

    NUM_OCTAVES = 4
    NUM_SCALES = 5
    N_RUNS = 3

    div2k_dir = os.path.join('/content/drive/MyDrive/Colab Notebooks', 'DIV2K_train_HR')
    OUTPUT_DIR = os.path.join('/content/drive/MyDrive/Colab Notebooks', 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    image_paths = sorted(glob.glob(os.path.join(div2k_dir, "*.png")))[:10]

    print("DIV2K dir:", div2k_dir)
    print("Found images:", len(image_paths))

    # Warm-up JIT
    dummy = np.zeros((64, 64), dtype=np.float32)
    _ = build_gaussian_pyramid_gpu_v1(dummy, num_octaves=1, num_scales=NUM_SCALES, sigma_base=1.6)
    cuda.synchronize()

    all_gpu_times = []
    all_cpu_times = []

    if not image_paths:
        print("Khong co anh PNG nao trong DIV2K_train_HR.")
    else:
        print(f"Benchmarking {len(image_paths)} images...")

    for i, img_path in enumerate(image_paths, 1):
        img_color = cv2.imread(img_path)
        if img_color is None:
            print(f"[WARN] Khong doc duoc: {img_path}")
            continue

        gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
        image = gray.astype(np.float32) / 255.0  # khop CPU ref

        # GPU
        gpu_times = []
        for _ in range(N_RUNS):
            t0 = time.perf_counter()
            _ = build_gaussian_pyramid_gpu_v1(
                image,
                num_octaves=NUM_OCTAVES,
                num_scales=NUM_SCALES,
                sigma_base=1.6
            )
            cuda.synchronize()
            gpu_times.append(time.perf_counter() - t0)
        avg_gpu = 1000 * np.mean(gpu_times)
        all_gpu_times.append(avg_gpu)

        # CPU ref
        cpu_times = []
        for _ in range(N_RUNS):
            t0 = time.perf_counter()
            _ = build_gaussian_pyramid_with_lib(
                image,
                num_octaves=NUM_OCTAVES,
                num_scales=NUM_SCALES,
                sigma_base=1.6
            )
            cpu_times.append(time.perf_counter() - t0)
        avg_cpu = 1000 * np.mean(cpu_times)
        all_cpu_times.append(avg_cpu)

        h, w = gray.shape
        print(
            f"[{i}/{len(image_paths)}] {os.path.basename(img_path)} ({w}x{h}) "
            f"| CPU: {avg_cpu:.1f} ms | GPU V1: {avg_gpu:.1f} ms | Speedup: {avg_cpu / avg_gpu:.2f}x"
        )

        # Chạy OpenCV SIFT trên ảnh để lưu kết quả keypoint
        kp, des, t_sift, scale = run_sift_opencv(gray, max_detect_dim=1200)
        name = os.path.splitext(os.path.basename(img_path))[0]
        out_path = os.path.join(OUTPUT_DIR, f"{name}_v1_result.png")
        img_kp = cv2.drawKeypoints(img_color, kp, None, color=(0, 255, 255), flags=cv2.DRAW_MATCHES_FLAGS_DEFAULT)
        cv2.imwrite(out_path, img_kp)

    if all_gpu_times:
        overall_cpu = float(np.mean(all_cpu_times))
        overall_gpu = float(np.mean(all_gpu_times))
        print("-" * 60)
        print(f"Overall CPU:   {overall_cpu:.1f} ms/image")
        print(f"Overall GPUV1: {overall_gpu:.1f} ms/image")
        print(f"Speedup:       {overall_cpu / overall_gpu:.2f}x")


if __name__ == "__main__":
    main()


  V1_gpu_naive.py -- GPU V1 (Naive Numba CUDA)
DIV2K dir: /content/drive/MyDrive/Colab Notebooks/DIV2K_train_HR
Found images: 10


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Benchmarking 10 images...
[1/10] 0001.png (2040x1404) | CPU: 412.2 ms | GPU V1: 96.6 ms | Speedup: 4.27x
[2/10] 0002.png (2040x1848) | CPU: 967.1 ms | GPU V1: 104.5 ms | Speedup: 9.25x
[3/10] 0003.png (2040x1356) | CPU: 371.2 ms | GPU V1: 78.1 ms | Speedup: 4.75x
[4/10] 0004.png (2040x1344) | CPU: 348.7 ms | GPU V1: 76.4 ms | Speedup: 4.57x
[5/10] 0005.png (1608x2040) | CPU: 701.5 ms | GPU V1: 93.2 ms | Speedup: 7.53x
[6/10] 0006.png (1356x2040) | CPU: 508.3 ms | GPU V1: 75.4 ms | Speedup: 6.74x
[7/10] 0007.png (2040x1356) | CPU: 453.9 ms | GPU V1: 77.2 ms | Speedup: 5.88x
[8/10] 0008.png (2040x1356) | CPU: 369.7 ms | GPU V1: 76.0 ms | Speedup: 4.86x
[9/10] 0009.png (2040x1524) | CPU: 423.9 ms | GPU V1: 85.1 ms | Speedup: 4.98x
[10/10] 0010.png (2040x1644) | CPU: 645.8 ms | GPU V1: 93.6 ms | Speedup: 6.90x
------------------------------------------------------------
Overall CPU:   520.2 ms/image
Overall GPUV1: 85.6 ms/image
Speedup:       6.08x
